In [56]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [45]:
# (No edit) Imports
import os
import json

import sys
from pathlib import Path
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from art.estimators.classification import PyTorchClassifier
from art.attacks.evasion import ProjectedGradientDescent
from art.attacks.evasion import FastGradientMethod
from art.attacks.evasion import SaliencyMapMethod
from art.attacks.evasion import CarliniL2Method
# from art.attacks.evasion import Carlin
from sklearn.preprocessing import MinMaxScaler

sys.path.append(str(Path.cwd().parents[2]))

import os

import torch
from utils.models import OBU
from utils.functions import get_windowed_data, load_model_checkpoint

from sklearn.metrics import precision_score, recall_score, f1_score

In [46]:
# Inputs
checkpoint_file="../../../saved_models/RandomPos-cenFL-norm-updated.ckpt"
data_file = "../../../data/RandomPos_0709.csv"
train_perc = 80

norm_trained = True
collapse_output = True # True if using JSMA (maybe CW????)


In [47]:
# (No edit) Load models, define wrappers
model = load_model_checkpoint(checkpoint_file, gpu=False)

class SequenceCrossEntropy(nn.Module):
    def __init__(self):
        super().__init__()
        self.loss = nn.CrossEntropyLoss()

    def forward(self, a, b):
        if a.dim() == 3:
            # sequence output: (batch, seq_len, num_classes)
            if b.dim() == 3:
                b = b.argmax(dim=-1)
            return self.loss(a.permute(0, 2, 1), b.long())
        else:
            # collapsed output: (batch, num_classes)
            if b.dim() == 2:
                b = b.argmax(dim=-1)
            return self.loss(a, b.long())


class NormalizedCfCWrapper(nn.Module):
    def __init__(self, modena_model, data_min, data_max):
        super().__init__()
        self.modena_model = modena_model
        if not norm_trained:
            # MUST wrap in torch.tensor() explicitly — do not assign raw numpy arrays
            self.register_buffer("data_min", torch.tensor(data_min, dtype=torch.float32))
            self.register_buffer("data_max", torch.tensor(data_max, dtype=torch.float32))

    def forward(self, x_normalized):        
        if not norm_trained:
            x_raw = x_normalized * (self.data_max - self.data_min) + self.data_min
        else:
            x_raw = x_normalized
        logits, _ = self.modena_model(x_raw)

        if collapse_output:
            return logits.mean(dim=1)
        else:
            return logits

GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
/home/sliu/Documents/GitHub/reu26/.venv/lib/python3.12/site-packages/pytorch_lightning/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Checkpoint path exists!


In [62]:
# (no edit) FUNCTION to set model variables
wrapped_model, criterion, optimizer, classifier = None, None, None, None
def set_model_vars (in_x_train):
    global wrapped_model, criterion, optimizer, classifier
    wrapped_model = NormalizedCfCWrapper(modena_model=model.model,
                                        data_min=in_x_train.amin(dim=(0,1)), # x_train.min(axis=(0,1)) in numpy
                                        data_max=in_x_train.amax(dim=(0,1)))
    criterion = SequenceCrossEntropy()
    optimizer = optim.Adam(
        wrapped_model.parameters(),
        lr=0.001
    )

    classifier = PyTorchClassifier(
        model=wrapped_model,
        loss=criterion,
        optimizer=optimizer,
        input_shape=(10, 7),
        nb_classes=20, # the range [0, 3]; WRONG: the number of unique classes in y_test, len(np.unique(y_test.numpy()))
        clip_values=(0.0, 1.0), # for normalized
        device_type="cpu"
    )

In [49]:
# (No edit) Load data for og model (very time consuming part)
(x_train, y_train), (x_test, y_test), fed_dataset = get_windowed_data(data_file, 
                                                                      normalize=norm_trained, 
                                                                      train_perc=train_perc)



In [50]:
# TEST: Test the model - ensure there's a high accuracy with the original model
set_model_vars(x_train)
print("TESTING |", f"norm_trained: {norm_trained} | collapse output {collapse_output}", "| NO wrapper")
out = model.test(x_test, y_test, mathy=True)

# # Test how replicable the results are
# not_same = 0
# for i in range(20):
#     print(f"=== {i+1}/20 ===")
#     out_test = model.test(x_test, y_test, mathy=True)
#     if out != out_test:
#         print("NOT SAME!")
#         not_same +=1
# print("# of times with distinct output:", not_same)

print(f"Out: {out}")

TESTING | norm_trained: True | collapse output True | NO wrapper
torch.Size([124774, 10, 20])
0.9921024389045001
0.9800296735905044
Model got 1237445/1247740 right.
Accuracy: 0.9917490823408723, Precision: 0.9921024389045001, Recall: 0.9800296735905044, F1 Score: 0.9860291034334886
877040, 70.29028483498165% Zeroes, 370700 Non Zero entries.
Out: (0.9860291034334886, 0.9800296735905044, 0.9921024389045001, 0.9917490823408723)


In [51]:
# IF NEEDED (if og model is was not norm_trained)- Load data at *normalized* 
if not norm_trained:
    print("Setting normalized data")
    (x_train, y_train), (x_test, y_test), fed_dataset = get_windowed_data(data_file, 
                                                                        normalize=True, 
                                                                        train_perc=train_perc)


In [52]:
# TEST: Benign test
set_model_vars(x_train)

benign_y_test = y_test.numpy()
if collapse_output:
    y_test_collapsed = y_test[:, 0].numpy()  # just take first timestep, since all are the same
    benign_y_test = y_test_collapsed

benign_predictions = classifier.predict(x_test, batch_size=64)
benign_pred_classes = np.argmax(benign_predictions, axis=-1)  # (N, seq_len)
true_flat = benign_y_test.flatten()
pred_flat = benign_pred_classes.flatten()

accuracy = np.sum(benign_pred_classes == benign_y_test) / benign_pred_classes.size
precision =  precision_score(true_flat, pred_flat, average="weighted", zero_division=0)
recall = recall_score(true_flat, pred_flat, average="weighted", zero_division=0)
f1 = f1_score(true_flat, pred_flat, average="weighted", zero_division=0)

print(f"Benign accuracy: {accuracy}")
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)

# # Test if scores are replicable
# not_same = 0
# for i in range(20):
#     print(f"=== {i+1}/20 ===")
#     benign_predictions = classifier.predict(x_test, batch_size=64)
#     benign_pred_classes = np.argmax(benign_predictions, axis=-1)  # (N, seq_len)
#     true_flat = y_test.flatten()
#     pred_flat = benign_pred_classes.flatten()

#     accuracy_test = np.sum(benign_pred_classes == y_test.numpy()) / benign_pred_classes.size
#     precision_test =  precision_score(true_flat, pred_flat, average="weighted", zero_division=0)
#     recall_test = recall_score(true_flat, pred_flat, average="weighted", zero_division=0)
#     f1_test = f1_score(true_flat, pred_flat, average="weighted", zero_division=0)

#     if accuracy == accuracy_test and precision == precision_test \
#         and recall == recall_test and f1 == f1_test:
#         pass
#     else:
#         print("NOT SAME! f1:", f1)

# print("# of distinct values:", not_same)

Benign accuracy: 0.9976998413130941
Precision: 0.9977073437342865
Recall: 0.9976998413130941
F1: 0.997697250661112


In [66]:
# Adversarial test function
def adv_test(end_index : int, path : str, filename : str, Attack, **kwargs):
    print(f"=== kwargs: {kwargs} ===")

    adv_y_test = y_test.numpy()
    if collapse_output:
        y_test_collapsed = y_test[:, 0].numpy()  # just take first timestep, since all are the same
        adv_y_test = y_test_collapsed
    adv_y_test = adv_y_test[:end_index]
    attack = Attack(classifier, **kwargs)
    
    x_test_adv = attack.generate(x=x_test.numpy()[:end_index])#, y=y_test[:fraction])
    adversarial_predictions = classifier.predict(x_test_adv)
    pred_classes = np.argmax(adversarial_predictions, axis=-1)  # (N, seq_len)
    adversarial_accuracy = np.sum(pred_classes == adv_y_test) / pred_classes.size
    print(f"Adversarial accuracy: {adversarial_accuracy}")

    true_flat = adv_y_test.flatten()
    pred_flat = pred_classes.flatten()
    precision = precision_score(true_flat, pred_flat, average="weighted", zero_division=0)
    recall = recall_score(true_flat, pred_flat, average="weighted", zero_division=0)
    f1 = f1_score(true_flat, pred_flat, average="weighted", zero_division=0)

    print("Precision:", precision)
    print("Recall:", recall)
    print("F1:", f1)

    # Save metrics to JSON in ./output
    os.makedirs(path, exist_ok=True)
    metrics = {
        "endIndex": end_index,
        "kwargs": kwargs,
        "accuracy": float(adversarial_accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
    }
    # output_path = f"{path}/adv_test_eps_{eps}.json"
    output_path = f"{path}/{filename}"
    with open(output_path, "w") as f:
        json.dump(metrics, f, indent=4)
    print(f"Saved metrics to {output_path}")

    return metrics

In [67]:
# FGSM - eps = 0.1
# PGD - eps = 0.1, 
# JSMA - swept gamma | collapsed_output = true, theta = amount perturbed, gamma = fraction of features perturbed

adv_test(end_index = 5000, # len(y_test.numpy()),
        path = "../adv_data/test",
        filename=f"adv.json",
        Attack=CarliniL2Method)

# CarliniL2Method()


=== kwargs: {} ===


KeyboardInterrupt: 

In [ ]:
import time 
set_model_vars(x_train)

# FGSM attack success rate (ASR): only counts malicious (label != 0) samples
# that get evaded to benign (predicted 0) after the attack.
def adv_asr(end_index: int, path : str, filename : str, Attack, **kwargs):
    start = time.time_ns()
    adv_y_test = y_test.numpy()
    if collapse_output:
        adv_y_test = y_test[:, 0].numpy()  # first timestep, all same
    adv_y_test = adv_y_test[:end_index]

    attack = Attack(classifier, **kwargs)
    x_test_adv = attack.generate(x=x_test.numpy()[:end_index])

    adversarial_predictions = classifier.predict(x_test_adv)
    pred_classes = np.argmax(adversarial_predictions, axis=-1)  # (N, seq_len) or (N,)

    true_flat = adv_y_test.flatten()
    pred_flat = pred_classes.flatten()

    malicious_mask = true_flat != 0
    n_malicious = int(malicious_mask.sum())
    if n_malicious == 0:
        print("No malicious (label != 0) samples in this slice.")
        return None

    evaded = int(np.sum(pred_flat[malicious_mask] == 0))
    asr = evaded / n_malicious

    print(f"Malicious samples: {n_malicious}")
    print(f"Evaded to benign: {evaded}")
    print(f"Attack Success Rate (malicious -> benign): {asr}")

    os.makedirs(path, exist_ok=True)
    metrics = {
        "endIndex": end_index,
        "attack": Attack.__name__,
        "kwargs": kwargs,
        "maliciousSamples": n_malicious,
        "evadedSamples": evaded,
        "attackSuccessRate": float(asr),
        "timeElapsedNs": time.time_ns() - start
    }
    # output_path = f"{path}/adv_test_eps_{eps}.json"
    output_path = f"{path}/{filename}"
    with open(output_path, "w") as f:
        json.dump(metrics, f, indent=4)
    print(f"Saved metrics to {output_path}")

    return asr

# for i in range(1, 11):
#     # eps = float(i/100) 
#     gamma = i/10
#     # print(gamma)
#     adv_asr(end_index=5000, # len(y_test.numpy()), 
#             path="../asr_data/jsma-norm_end-5000", 
#             filename = f"asr_data_gamma_{gamma}.json",
#             Attack = SaliencyMapMethod,
#             gamma=gamma)
adv_asr(end_index=5000, # len(y_test.numpy()), 
            path="../asr_data/cw-norm", 
            filename = f"asr_data.json",
            Attack = CarliniL2Method)

C&W L_2:   9%|▉         | 451/5000 [08:47<1:29:11,  1.18s/it]